In [2]:
import requests
import json
import pandas as pd
import os
import tqdm as tqdm
import requests
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote_plus
import rdflib
import os
from rdflib.namespace import RDF, DC, Namespace
# import xml.etree.ElementTree as ET
from lxml import etree
import random
from lxml import etree as ET

In [2]:
import os

def get_directory_size_gb(directory):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(directory):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            # Skip if it is symbolic link
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    # Convert bytes to GB
    return total_size / (1024 ** 3)

# Example usage
total_files = get_directory_size_gb('/home/sbasir/Thesis/Thesis/collected3')
print(f"Total size of the directory: {total_files:.2f} GB")

Total size of the directory: 11.24 GB


In [25]:
import os

def count_files(directory_path):
    file_count = 0
    for root, dirs, files in os.walk(directory_path):
        file_count += len(files)
    return file_count

# Example usage:
total_files = count_files('/home/sbasir/Thesis/Thesis/collected3')
total_files2 = count_files('/home/sbasir/Thesis/Thesis/collected3_split/25_percent')
total_files3 = count_files('/home/sbasir/Thesis/Thesis/collected3_split/25_2_percent/75_100_percentile')
total_files4 = count_files('/home/sbasir/Thesis/Thesis/collected3_split/25_3_percent/75_100_percentile')
total_files5 = count_files('/home/sbasir/Thesis/Thesis/collected3_split/25_4_percent/75_100_percentile')

print(total_files, total_files2, total_files3, total_files4, total_files5)

4072603 1018150 1018151 1018151 1018151


In [26]:
def get_subdirectories(directory_path):
    subdirs = [d for d in os.listdir(directory_path) if os.path.isdir(os.path.join(directory_path, d))]
    return subdirs

# results1 = get_subdirectories('/home/sbasir/Thesis/Thesis/collected3_split/75_percent')
# results2 = get_subdirectories('/home/sbasir/Thesis/Thesis/collected3_split/50_percent')
# results3 = get_subdirectories('/home/sbasir/Thesis/Thesis/collected3_split/25_percent')

idk1 = get_subdirectories('/home/sbasir/Thesis/Thesis/collected3_split/25_percent')
idk2 = get_subdirectories('/home/sbasir/Thesis/Thesis/collected3_split/25_2_percent/75_100_percentile')
idk3 = get_subdirectories('/home/sbasir/Thesis/Thesis/collected3_split/25_3_percent/75_100_percentile')
idk4 = get_subdirectories('/home/sbasir/Thesis/Thesis/collected3_split/25_4_percent/75_100_percentile')



In [31]:
# check which files are in common
common = set(idk1) & set(idk2) & set(idk3) & set(idk4)
print(common)

# check which files are not in common
not_common = set(idk1) ^ set(idk2) ^ set(idk3) ^ set(idk4)
print(not_common)

print(len(common), len(not_common))

set()
{'2048608', '2048425', '9200321', '606', '2058643', '15507', '9200143', '859', '934', '9200436', '10907', '2022093', '00741', '9200166', '0940401', '2022412', '91693', '470', '218', '2025902', '91655', '91642', '11647', '2021612', '2058619', '9200577', '863', '645', '633', '463', '2058617', '2064101', '306', '667', '0943101', '837', '191', '2022717', '454', '9200245', '9200347', '2064130', '666', '2022368', '478', '2020710', '2064202', '9200133', '2026117', '733', '9200441', '2048424', '9200440', '9200171', '369', '249', '185', '2032010', '2048711', '00738', '1030', '235', '9200268', '92093', '799', '2022411', '9200114', '670', '9200476', '617', '541', '9200200', '9200235', '9200353', '9200392', '2022421', '2021602', '9200373', '9200232', '0943115', '2021802', '2022024', '2063630', '0943110', '884', '00720', '283', '776', '9200355', '1031', '2063617', '2021627', '803', '00719', '9200438', '794', '9200318', '780', '757', '739', '92065', '9200414', '07931', '806', '356', '92076', '

In [6]:
import csv
def read_csv_data(csv_path):
    with open('/home/sbasir/Thesis/Thesis/EDP/sample_data/datasets.csv', newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)

        limit = 300000

        # Extract 'ids' and 'lang' and save them in a dictionary, but only for rows where 'ndocs' is less than limit
        data_dict = {row['ids'] + '.zip': (row['lang'], row['ndocs']) for row in reader} #if int(row['ndocs']) < limit}

        csvfile.seek(0)  # Rewind the CSV file
        next(reader)  # Skip the header
        data_dict_alt = {row['ids'] + '.zip': (row['lang'], row['ndocs']) for row in reader if int(row['ndocs']) > limit}

        csvfile.seek(0)  # Rewind the CSV file
        next(reader)  # Skip the header

        # Extract only 'ids' in a separate list, but only for rows where 'ndocs' is less than limit
        data_ids = [row['ids'] for row in reader] #if int(row['ndocs']) < limit]

        csvfile.seek(0)  # Rewind the CSV file
        next(reader)  # Skip the header

        data_ids_alt = [row['ids'] + '.zip' for row in reader if int(row['ndocs']) > limit]

        return data_ids

data_ids = read_csv_data('/home/sbasir/Thesis/Thesis/EDP/sample_data/datasets.csv')

In [7]:
not_downloaded = list(set(data_ids) - set(results))
print(not_downloaded)

['1145', '1051', '1169', '1128', '2021006', '1129', '1158', '1005', '1131', '2022503', '1181', '1095', '824', '1036', '1011', '1114', '1052', '1082', '52', '996', '1171', '1038', '73', '1166', '579', '1003', '955', '1179', '1002', '9200214', '1010', '2051917', '1013', '1008', '344', '346', '1200', '544', '931', '323', '1079', '1142', '1061', '1135', '1112', '410', '1009', '990', '1055', '2048375', '876', '2021112', '887', '874', '1172', '1075', '1190', '1197', '602', '880', '1083', '1081', '2058703', '895', '75', '983', '2048443', '2022710', '538', '1140', '0943105', '2058704', '897', '1096', '1124', '601', '1132', '412', '1163', '1034', '1063', '1175', '1160', '1025', '1115', '2064105', '1056', '1123', '1014', '994', '1111', '1116', '9200221', '1118', '303', '900', '535', '1121', '1168', '1177', '348', '1092', '2020108', '1084', '1130', '1012', '72', '877', '1035', '966', '879', '1024', '1193', '1113', '1043', '9200225', '1004', '953', '2021001', '0940430', '957', '963', '1199', '534'

In [8]:
remaining = [item + ".csv" for item in not_downloaded]
print(len(remaining))

198


In [9]:
import os

remaining_again = []
def check_files_exist(filenames, directory_path):
    # Loop through each filename and check if it exists in the specified directory
    for filename in filenames:
        if os.path.isfile(os.path.join(directory_path, filename)):
            print(f"File found: {filename}")
            remaining_again.append(filename.replace(".csv", ".zip"))
    
    return remaining_again

# Example usage:
filenames = remaining
directory_path = "/home/sbasir/Thesis/Thesis/EDP/sample_data/translations"
result = check_files_exist(filenames, directory_path)

# Print the result
print(result)

File found: 824.csv
File found: 2051917.csv
File found: 344.csv
File found: 2021112.csv
File found: 2022710.csv
File found: 538.csv
File found: 0943105.csv
File found: 9200221.csv
File found: 303.csv
File found: 2048427.csv
File found: 2021606.csv
File found: 2024014.csv
File found: 2022720.csv
File found: 2058632.csv
File found: 315.csv
['824.zip', '2051917.zip', '344.zip', '2021112.zip', '2022710.zip', '538.zip', '0943105.zip', '9200221.zip', '303.zip', '2048427.zip', '2021606.zip', '2024014.zip', '2022720.zip', '2058632.zip', '315.zip']


In [2]:
import pandas as pd

def load_and_sort_by_ndocs(file_path):
    # Read the CSV file into a DataFrame
    df = pd.read_csv(file_path)
    
    # Sort the DataFrame by 'ndocs' column
    sorted_df = df.sort_values(by='ndocs', ascending=False)
    
    return sorted_df

def print_sorted_data(sorted_df):
    # Print the sorted DataFrame
    print(sorted_df)

# Example usage
file_path = '/home/sbasir/Thesis/Thesis/EDP/sample_data/datasets.csv'  # Replace with your actual file path
sorted_df = load_and_sort_by_ndocs(file_path)
display(sorted_df[:50])


,ids,name,lang,ndocs
1331,9200365,9200365_Ag_EU_TEL_a0142_Gallica,fr,1191745
1326,9200359,9200359_Ag_EU_TEL_a0601_Newspapers_Netherlands,nl,747773
1404,9200479,9200479_NLPoland,pl,644682
1343,9200384,9200384_Ag_EU_TEL_a0613_Newspapers_ONB,de,629498
822,401,401_Muuseumid,et,613372
243,11614,11614_Royal_Botanic_Gardens_Kew,en,595140
546,2048087,2048087_Ag_EU_AthenaPlus_CollectionsTrust,en,584554
335,2020702,2020702_Ag_EU_CARARE_SNHB,sv,547566
646,2058632,2058632_Ag_EU_LoCloud_RCE,nl,526228
547,2048128,2048128_Ag_HU_MaNDA_OAI,hu,523029


In [112]:
# directory = '/Users/suhaibbasir/Documents/CS/MSc/Thesis/Thesis/EDP/test'  # Change this to your directory containing RDF/XML files
# Define namespace mappings
# Define the namespaces
namespaces = {
    'rdf': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
    'dc': 'http://purl.org/dc/elements/1.1/',
    'dcterms': 'http://purl.org/dc/terms/',
    'edm': 'http://www.europeana.eu/schemas/edm/',
    'skos': 'http://www.w3.org/2004/02/skos/core#',
    'foaf': 'http://xmlns.com/foaf/0.1/',
    'ore': 'http://www.openarchives.org/ore/terms/',
    'dqv': 'http://www.w3.org/ns/dqv#',
    'oa': 'http://www.w3.org/ns/oa#'
}

# Define functions to find elements
def find_single_element_text(tree, xpath_query):
    element = tree.find(xpath_query, namespaces)
    if element is not None:
        resource = element.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource')
        if resource:
            # print("Resource found for:", xpath_query)
            # Find the corresponding element using rdf:about attribute
            linked_element = tree.find(f".//*[@rdf:about='{resource}']", namespaces)
            if linked_element is not None:
                # Extract text content of skos:prefLabel within the linked element
                label = linked_element.find('skos:prefLabel', namespaces)
                if label is not None:
                    return label.text
            return resource  # Fallback to the resource URI if linked element not found
        else:
            return element.text
    return None

def find_multiple_elements_text(tree, xpath_query):
    elements = tree.findall(xpath_query, namespaces)
    results = {}
    for element in elements:
        if elements:
            for element in elements:
                # Check if the element has an xml:lang attribute
                language_tag = element.get('{http://www.w3.org/XML/1998/namespace}lang', 'default')  # 'default' if no lang

                # Check if the element has a reference to a resource
                resource_ref = element.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource')

                if resource_ref:
                    # Find the element with rdf:about matching the resource reference
                    resource_element = tree.xpath(f"//*[@rdf:about='{resource_ref}']", namespaces=namespaces)
                    if resource_element:
                        # Get the skos:prefLabel and skos:altLabel from the resource element
                        pref_labels = resource_element[0].xpath("skos:prefLabel", namespaces=namespaces)
                        alt_labels = resource_element[0].xpath("skos:altLabel", namespaces=namespaces)

                        # Combine both prefLabel and altLabel in one list
                        all_labels = pref_labels + alt_labels

                        for label in all_labels:
                            # Get the label text and the xml:lang attribute
                            label_text = label.text
                            label_lang = label.get('{http://www.w3.org/XML/1998/namespace}lang', 'default')

                            # Append the text to the corresponding language in the results dictionary
                            if label_lang in results:
                                results[label_lang].append(label_text)
                            else:
                                results[label_lang] = [label_text]
                else:
                    # Add the text content of the element if no resource reference or labels are found
                    if language_tag in results:
                        results[language_tag].append(element.text)
                    else:
                        results[language_tag] = [element.text]
    return results


def find_tier_information(tree, namespaces):
    # Initialize variables to store tier information
    content_tier = None
    metadata_tier = None

    # Find all hasBody elements
    has_body_elements = tree.findall('.//oa:hasBody', namespaces)

    for element in has_body_elements:
        resource = element.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource', '')
        
        # Check for content tier
        if 'contentTier' in resource:
            content_tier = resource.split('contentTier')[-1]
        
        # Check for metadata tier
        elif 'metadataTier' in resource:
            metadata_tier = resource.split('metadataTier')[-1]

    return content_tier, metadata_tier

def find_with_condition(condition, search, filename, tree, namespaces, lang=None):
    # Find all <ore:Proxy> elements with rdf:about containing the condition
    paths = tree.xpath(f"//ore:Proxy[contains(@rdf:about, '{condition}')]", namespaces=namespaces)
    
    # Initialize a dictionary to store results based on language
    results = {}

    languages = ["en", "bg", "cs", "da", "de", "es", "et", "fi", "fr", "hr", "hu", "it", "lt", "nl", "pl", "pt", "ro", "sk", "sl", "sv"]
    
    # Check if we found the <ore:Proxy> elements
    if paths:
        for path in paths:
            # Build the XPath query with an optional language filter
            if lang:
                elements = path.xpath(f"{search}[@xml:lang='{lang}']", namespaces=namespaces)
            else:
                elements = path.xpath(f"{search}", namespaces=namespaces)
                # Check if any of these elements have lang='en'
                # en_elements = path.xpath(f"{search}[@xml:lang='en']", namespaces=namespaces)
                # if en_elements:
                #     for en_element in en_elements:print(en_element.text)
                #     print("AAAA")
                #     continue  # Skip this iteration if any elements with lang='en' are found

            if elements:
                for element in elements:
                    # Check if the element has an xml:lang attribute
                    language_tag = element.get('{http://www.w3.org/XML/1998/namespace}lang', 'default')  # 'default' if no lang

                    # Check if the element has a reference to a resource
                    resource_ref = element.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource')
                        
                    if language_tag == "en" and not resource_ref and lang is None and condition == "/europeana":
                        continue

                    if resource_ref:
                        # Find the element with rdf:about matching the resource reference
                        resource_element = tree.xpath(f"//*[@rdf:about='{resource_ref}']", namespaces=namespaces)
                        if resource_element:
                            # Get the skos:prefLabel and skos:altLabel from the resource element
                            pref_labels = resource_element[0].xpath("skos:prefLabel", namespaces=namespaces)
                            alt_labels = resource_element[0].xpath("skos:altLabel", namespaces=namespaces)

                            # Combine both prefLabel and altLabel in one list
                            all_labels = pref_labels + alt_labels

                            for label in all_labels:
                                # Get the label text and the xml:lang attribute
                                label_text = label.text
                                label_lang = label.get('{http://www.w3.org/XML/1998/namespace}lang', 'default')

                                if label_lang in languages or label_lang == "default":
                                    print("idk bruh :", label_lang in languages)
                                    # Append the text to the corresponding language in the results dictionary
                                    if label_lang in results:
                                        results[label_lang].append(label_text)
                                    else:
                                        results[label_lang] = [label_text]
                    else:
                        # Add the text content of the element if no resource reference or labels are found
                        if language_tag in languages or language_tag == "default":
                            if language_tag in results:
                                results[language_tag].append(element.text)
                            else:
                                results[language_tag] = [element.text]

    else:
        print(f'No matching ore:Proxy element found in {filename}')
    
    # Return the dictionary of results, or None if no matches
    return results if results else None


def check_if_translated(europeana_id, subdirectory):
    try:
        # Extract the last part of the europeana_id
        last_part = europeana_id.split('/')[-1]
        
        # Construct the file path for the translation CSV
        translation_subdirectory = subdirectory + '.csv'
        csv_path = f'/home/sbasir/Thesis/Thesis/EDP/sample_data/translations/{translation_subdirectory}'
        
        # Check if the CSV file exists
        if not os.path.exists(csv_path):
            # Return False if the file does not exist
            print("does not exist")
            return False
        
        # Load the CSV file
        translation_csv = pd.read_csv(csv_path)
        
        # Check if the last_part is in the 'item_id' column of the DataFrame
        translated = last_part in translation_csv['item_id'].values
        
        # Return True if found, False otherwise
        return translated

    except FileNotFoundError:
        # If the file is not found, print an error message and return False
        print(f"File not found: {csv_path}")
        return False

    except Exception as e:
        # Catch all other exceptions, print an error message, and return False
        print(f"Error checking translation for {europeana_id}: {e}")
        return False

def generate_solr_xml(data):
    solr_docs = "<add>"
    
    # Add the top-level fields first (e.g., dcterms:modified, edm:type)
    solr_docs += f'<field name="europeana_id">{data["europeana_id"]}</field>'
    solr_docs += f'<field name="timestamp_update">{data["dcterms:modified"]}</field>'
    solr_docs += f'<field name="edm:type">{data["edm:type"]}</field>'
    solr_docs += f'<field name="content_tier">{data["contentTier"]}</field>'
    solr_docs += f'<field name="metadata_tier">{data["metadataTier"]}</field>'


    # Function to add fields with language variants
    def add_field_with_lang(field_name, lang_values):
        nonlocal solr_docs
        if lang_values:
            solr_docs += f'<field name="{field_name}">'
            for lang, values in lang_values.items():
                for value in values:
                    if lang == 'default':
                        solr_docs += f'<value>{value}</value>'  # No language tag
                    else:
                        solr_docs += f'<value lang="{lang}">{value}</value>'  # With language tag
            solr_docs += '</field>'

    # Add provided_data section
    solr_docs += "<provided_data>"
    provided_data = data.get('provided_data', {})
    for field, values in provided_data.items():
        if isinstance(values, dict):  # Handle language variants
            add_field_with_lang(field, values)
        elif values:  # Handle fields without language variants
            solr_docs += f'<field name="{field}">{values}</field>'
    solr_docs += "</provided_data>"

    # Add enriched_data section
    solr_docs += "<enriched_data>"
    enriched_data = data.get('enriched_data', {})
    for field, values in enriched_data.items():
        if isinstance(values, dict):  # Handle language variants
            add_field_with_lang(field, values)
    solr_docs += "</enriched_data>"

    # Add translated_data section
    solr_docs += "<translated_data>"
    translated_data = data.get('translated_data', {})
    for field, values in translated_data.items():
        if isinstance(values, dict):  # Handle language variants
            add_field_with_lang(field, values)
    solr_docs += "</translated_data>"

    solr_docs += "</add>"
    return solr_docs


def parse_rdf_files(directory, subdirectory):

    docs = []
    docs_alt = []

    ## TODO: IMPLEMENT THE CODE FOR LOOKING INTO THE RDF:RESOURCE ATTRIBUTE

    # Get all files in the directory that end with '.rdf' or '.xml'
    all_files = [filename for filename in os.listdir(directory) if filename.endswith('.rdf') or filename.endswith('.xml')]

    percentage = 1
    
    # Calculate the number of files to sample (10% of total)
    sample_size = max(1, int(len(all_files) * percentage))  # Ensure at least one file is selected if len(all_files) < 10

    # Randomly sample 10% of the files
    sampled_files = random.sample(all_files, sample_size)

    # Iterate over each file in the directory
    for filename in sampled_files:
        # print(filename)
        file_path = os.path.join(directory, filename)
        
        try:
            # Load the XML file
            tree = etree.parse(file_path)
            # print(f'Parsed: {filename}')

            # checks for sampled data:
            # check 1: based on content tier - if 0 do not include
            # check 2: based on translations - if english include, it not english include if translated

            content_tier, metadata_tier = find_tier_information(tree, namespaces)
            europeana_id = find_single_element_text(tree, './/ore:proxyIn')

            isTranslated = check_if_translated(europeana_id, subdirectory)

            if not isTranslated:
                print("IS NOT TRANSLATED")
                language = find_single_element_text(tree, './/edm:EuropeanaAggregation/edm:language')
                print(language)
                # if language != 'en':
                #     print("SKIPPING")
                #     continue
            else:
                print("is translated")

            if content_tier == 0:
                print("SKIPPING")
                continue
            else:
                print("content is fine")
            
            print("passed checks")
            # extract data

            europeana_id = '/'.join(europeana_id.rsplit('/', 2)[-2:])
            europeana_id = '/'+europeana_id

            data = {
                'europeana_id': europeana_id,
                'dcterms:modified': find_single_element_text(tree, './/dcterms:modified'),
                'edm:type': find_single_element_text(tree, './/edm:type'),
                'contentTier': content_tier, 
                'metadataTier': metadata_tier,
                "provided_data": {
                    'edm:dataProvider': find_multiple_elements_text(tree, './/ore:Aggregation/edm:dataProvider'),
                    'edm:intermediateProvider': find_multiple_elements_text(tree, './/ore:Aggregation/edm:intemediateProvider'),
                    'edm:provider': find_multiple_elements_text(tree, './/ore:Aggregation/edm:provider'),
                    'dc:contributor': find_with_condition("/provider", ".//dc:contributor", filename, tree, namespaces),
                    'dc:coverage': find_with_condition("/provider", ".//dc:coverage", filename, tree, namespaces),
                    'dc:creator': find_with_condition("/provider", ".//dc:creator", filename, tree, namespaces),
                    'dc:date': find_with_condition("/provider", ".//dc:date", filename, tree, namespaces),
                    'dc:description': find_with_condition("/provider", ".//dc:description", filename, tree, namespaces),
                    'dc:format': find_with_condition("/provider", ".//dc:format", filename, tree, namespaces),
                    'dc:language': find_with_condition("/provider", ".//dc:language", filename, tree, namespaces),
                    'dc:publisher': find_with_condition("/provider", ".//dc:publisher", filename, tree, namespaces),
                    'dc:source': find_with_condition("/provider", ".//dc:source", filename, tree, namespaces),
                    'dc:subject': find_with_condition("/provider", ".//dc:subject", filename, tree, namespaces),
                    'dc:title': find_with_condition("/provider", ".//dc:title", filename, tree, namespaces),
                    'dc:type': find_with_condition("/provider", ".//dc:type", filename, tree, namespaces),
                    'dcterms:alternative': find_with_condition("/provider", ".//dcterms:alternative", filename, tree, namespaces),
                    'dcterms:created': find_with_condition("/provider", ".//dcterms:created", filename, tree, namespaces),
                    'dcterms:issued': find_with_condition("/provider", ".//dcterms:issued", filename, tree, namespaces),
                    'dcterms:medium': find_with_condition("/provider", ".//dcterms:medium", filename, tree, namespaces),
                    'dcterms:provenance': find_with_condition("/provider", ".//dcterms:provenance", filename, tree, namespaces),
                    'dcterms:spatial': find_with_condition("/provider", ".//dcterms:spatial", filename, tree, namespaces),
                    'dcterms:temporal': find_with_condition("/provider", ".//dcterms:temporal", filename, tree, namespaces),
                    'edm:currentLocation': find_with_condition("/provider", ".//edm:currentLocation", filename, tree, namespaces),
                },
                "enriched_data": {
                    "dc:contributor": find_with_condition("/europeana", ".//dc:contributor", filename, tree, namespaces),
                    "dc:coverage": find_with_condition("/europeana", ".//dc:coverage", filename, tree, namespaces),
                    "dc:creator": find_with_condition("/europeana", ".//dc:creator", filename, tree, namespaces),
                    "dc:date": find_with_condition("/europeana", ".//dc:date", filename, tree, namespaces),
                    "dc:description": find_with_condition("/europeana", ".//dc:description", filename, tree, namespaces),
                    "dc:format": find_with_condition("/europeana", ".//dc:format", filename, tree, namespaces),
                    'dc:language': find_with_condition("/europeana", ".//dc:language", filename, tree, namespaces),
                    'dc:publisher': find_with_condition("/europeana", ".//dc:publisher", filename, tree, namespaces),
                    'dc:source': find_with_condition("/europeana", ".//dc:source", filename, tree, namespaces),
                    'dc:subject': find_with_condition("/europeana", ".//dc:subject", filename, tree, namespaces),
                    'dc:title': find_with_condition("/europeana", ".//dc:title", filename, tree, namespaces),
                    'dc:type': find_with_condition("/europeana", ".//dc:type", filename, tree, namespaces),
                    'dcterms:alternative': find_with_condition("/europeana", ".//dcterms:alternative", filename, tree, namespaces),
                    'dcterms:created': find_with_condition("/europeana", ".//dcterms:created", filename, tree, namespaces),
                    'dcterms:issued': find_with_condition("/europeana", ".//dcterms:issued", filename, tree, namespaces),
                    'dcterms:medium': find_with_condition("/europeana", ".//dcterms:medium", filename, tree, namespaces),
                    'dcterms:provenance': find_with_condition("/europeana", ".//dcterms:provenance", filename, tree, namespaces),
                    'dcterms:spatial': find_with_condition("/europeana", ".//dcterms:spatial", filename, tree, namespaces),
                    'dcterms:temporal': find_with_condition("/europeana", ".//dcterms:temporal", filename, tree, namespaces),
                    'edm:currentLocation': find_with_condition("/europeana", ".//edm:currentLocation", filename, tree, namespaces),
                },
                "translated_data":{
                    "dc:contributor": find_with_condition("/europeana", ".//dc:contributor", filename, tree, namespaces, lang='en'),
                    "dc:coverage": find_with_condition("/europeana", ".//dc:coverage", filename, tree, namespaces, lang='en'),
                    "dc:creator": find_with_condition("/europeana", ".//dc:creator", filename, tree, namespaces, lang='en'),
                    "dc:date": find_with_condition("/europeana", ".//dc:date", filename, tree, namespaces, lang='en'),
                    "dc:description": find_with_condition("/europeana", ".//dc:description", filename, tree, namespaces, lang='en'),
                    "dc:format": find_with_condition("/europeana", ".//dc:format", filename, tree, namespaces, lang='en'),
                    'dc:language': find_with_condition("/europeana", ".//dc:language", filename, tree, namespaces, lang='en'),
                    'dc:publisher': find_with_condition("/europeana", ".//dc:publisher", filename, tree, namespaces, lang='en'),
                    'dc:source': find_with_condition("/europeana", ".//dc:source", filename, tree, namespaces, lang='en'),
                    'dc:subject': find_with_condition("/europeana", ".//dc:subject", filename, tree, namespaces, lang='en'),
                    'dc:title': find_with_condition("/europeana", ".//dc:title", filename, tree, namespaces, lang='en'),
                    'dc:type': find_with_condition("/europeana", ".//dc:type", filename, tree, namespaces, lang='en'),
                    'dcterms:alternative': find_with_condition("/europeana", ".//dcterms:alternative", filename, tree, namespaces, lang='en'),
                    'dcterms:created': find_with_condition("/europeana", ".//dcterms:created", filename, tree, namespaces, lang='en'),
                    'dcterms:issued': find_with_condition("/europeana", ".//dcterms:issued", filename, tree, namespaces, lang='en'),
                    'dcterms:medium': find_with_condition("/europeana", ".//dcterms:medium", filename, tree, namespaces, lang='en'),
                    'dcterms:provenance': find_with_condition("/europeana", ".//dcterms:provenance", filename, tree, namespaces, lang='en'),
                    'dcterms:spatial': find_with_condition("/europeana", ".//dcterms:spatial", filename, tree, namespaces, lang='en'),
                    'dcterms:temporal': find_with_condition("/europeana", ".//dcterms:temporal", filename, tree, namespaces, lang='en'),
                    'edm:currentLocation': find_with_condition("/europeana", ".//edm:currentLocation", filename, tree, namespaces, lang='en'),
                }
            }

            print("data type is: ", type(data))
            print(data)

            solr_docs = "<add>"
            # Build Solr document
            solr_docs += f"""
            <doc>
                <field name="europeana_id">{europeana_id}</field>
                <field name="timestamp_update">{data['dcterms:modified']}</field>
                <field name="edm_type">{data['edm:type']}</field>
                <field name="content_tier">{content_tier}</field>
                <field name="metadata_tier">{metadata_tier}</field>
                <provided_data>
                    <field name="data_provider">{data['provided_data']['edm:dataProvider']}</field>
                    <field name="intermediate_provider">{data['provided_data']['edm:intermediateProvider']}</field>
                    <field name="provider">{data['provided_data']['edm:provider']}</field>
                    <field name="dc_contributor">{data['provided_data']['dc:contributor']}</field>
                    <field name="dc_coverage">{data['provided_data']['dc:coverage']}</field>
                    <field name="dc_creator">{data['provided_data']['dc:creator']}</field>
                    <field name="dc_date">{data['provided_data']['dc:date']}</field>
                    <field name="dc_description">{data['provided_data']['dc:description']}</field>
                    <field name="dc_format">{data['provided_data']['dc:format']}</field>
                    <field name="dc_language">{data['provided_data']['dc:language']}</field>
                    <field name="dc_publisher">{data['provided_data']['dc:publisher']}</field>
                    <field name="dc_source">{data['provided_data']['dc:source']}</field>
                    <field name="dc_subject">{data['provided_data']['dc:subject']}</field>
                    <field name="dc_title">{data['provided_data']['dc:title']}</field>
                    <field name="dc_type">{data['provided_data']['dc:type']}</field>
                    <field name="dcterms_alternative">{data['provided_data']['dcterms:alternative']}</field>
                    <field name="dcterms_created">{data['provided_data']['dcterms:created']}</field>
                    <field name="dcterms_issued">{data['provided_data']['dcterms:issued']}</field>
                    <field name="dcterms_medium">{data['provided_data']['dcterms:medium']}</field>
                    <field name="dcterms_provenance">{data['provided_data']['dcterms:provenance']}</field>
                    <field name="dcterms_spatial">{data['provided_data']['dcterms:spatial']}</field>
                    <field name="dcterms_temporal">{data['provided_data']['dcterms:temporal']}</field>
                    <field name="edm_currentLocation">{data['provided_data']['edm:currentLocation']}</field>
                </provided_data>
                <enriched_data>
                    <field name="dc_contributor">{data['enriched_data']['dc:contributor']}</field>
                    <field name="dc_coverage">{data['enriched_data']['dc:coverage']}</field>
                    <field name="dc_creator">{data['enriched_data']['dc:creator']}</field>
                    <field name="dc_date">{data['enriched_data']['dc:date']}</field>
                    <field name="dc_description">{data['enriched_data']['dc:description']}</field>
                    <field name="dc_format">{data['enriched_data']['dc:format']}</field>
                    <field name="dc_language">{data['enriched_data']['dc:language']}</field>
                    <field name="dc_publisher">{data['enriched_data']['dc:publisher']}</field>
                    <field name="dc_source">{data['enriched_data']['dc:source']}</field>
                    <field name="dc_subject">{data['enriched_data']['dc:subject']}</field>
                    <field name="dc_title">{data['enriched_data']['dc:title']}</field>
                    <field name="dc_type">{data['enriched_data']['dc:type']}</field>
                    <field name="dcterms_alternative">{data['enriched_data']['dcterms:alternative']}</field>
                    <field name="dcterms_created">{data['enriched_data']['dcterms:created']}</field>
                    <field name="dcterms_issued">{data['enriched_data']['dcterms:issued']}</field>
                    <field name="dcterms_medium">{data['enriched_data']['dcterms:medium']}</field>
                    <field name="dcterms_provenance">{data['enriched_data']['dcterms:provenance']}</field>
                    <field name="dcterms_spatial">{data['enriched_data']['dcterms:spatial']}</field>
                    <field name="dcterms_temporal">{data['enriched_data']['dcterms:temporal']}</field>
                    <field name="edm_currentLocation">{data['enriched_data']['edm:currentLocation']}</field>
                </enriched_data>
                <translated_data>
                        <field name="dc_contributor">{data['translated_data']['dc:contributor']}</field>
                    <field name="dc_coverage">{data['translated_data']['dc:coverage']}</field>
                    <field name="dc_creator">{data['translated_data']['dc:creator']}</field>
                    <field name="dc_date">{data['translated_data']['dc:date']}</field>
                    <field name="dc_description">{data['translated_data']['dc:description']}</field>
                    <field name="dc_format">{data['translated_data']['dc:format']}</field>
                    <field name="dc_language">{data['translated_data']['dc:language']}</field>
                    <field name="dc_publisher">{data['translated_data']['dc:publisher']}</field>
                    <field name="dc_source">{data['translated_data']['dc:source']}</field>
                    <field name="dc_subject">{data['translated_data']['dc:subject']}</field>
                    <field name="dc_title">{data['translated_data']['dc:title']}</field>
                    <field name="dc_type">{data['translated_data']['dc:type']}</field>
                    <field name="dcterms_alternative">{data['translated_data']['dcterms:alternative']}</field>
                    <field name="dcterms_created">{data['translated_data']['dcterms:created']}</field>
                    <field name="dcterms_issued">{data['translated_data']['dcterms:issued']}</field>
                    <field name="dcterms_medium">{data['translated_data']['dcterms:medium']}</field>
                    <field name="dcterms_provenance">{data['translated_data']['dcterms:provenance']}</field>
                    <field name="dcterms_spatial">{data['translated_data']['dcterms:spatial']}</field>
                    <field name="dcterms_temporal">{data['translated_data']['dcterms:temporal']}</field>
                    <field name="edm_currentLocation">{data['translated_data']['edm:currentLocation']}</field>
                </translated_data>
            </doc>
            """
            solr_docs += "</add>"
            docs.append(solr_docs)

            solr_doc_alt = generate_solr_xml(data)
            docs_alt.append(solr_doc_alt)
            
        except Exception as e:
            print(f"Failed to parse {filename}: {e}")
    
    # # Close the Solr document
    # solr_docs += "</add>"
    # docs.append(solr_docs)
    
    # Return the compiled Solr XML document
    return docs, docs_alt

def write_data(data, output_directory):
    # Ensure the output directory exists
    os.makedirs(output_directory, exist_ok=True)
    
    # Initialize a counter for file names
    doc_counter = 1

    # Iterate over each XML document in the data list
    for xml_doc in data:
        # Define the output file name
        output_file = os.path.join(output_directory, f"{doc_counter}.xml")

        # Write the data to the output file
        with open(output_file, 'w') as file:
            file.write(xml_doc)
        
        # print(f"Data written to {output_file}")

        # Increment the counter for the next file
        doc_counter += 1

def remove_duplicates(xml_data):
    # Parse the XML data
    root = ET.fromstring(xml_data)

    # Initialize a set to track unique europeana_id
    seen_ids = set()
    unique_docs = []

    # Iterate over each document and filter out duplicates
    for doc in root.findall(".//doc"):
        europeana_id = doc.find(".//field[@name='europeana_id']").text
        if europeana_id not in seen_ids:
            seen_ids.add(europeana_id)
            unique_docs.append(doc)

    # Build a new XML tree with unique documents
    new_root = ET.Element("add")
    for doc in unique_docs:
        new_root.append(doc)

    # Convert the tree back to a string
    new_xml_data = ET.tostring(new_root, encoding='unicode')
    return new_xml_data

def remove_none(xml_doc):
    # Parse the XML document using lxml
    root = ET.fromstring(xml_doc)

    # Iterate over all <doc> elements
    for doc in root.xpath(".//doc"):
        # Find all <field> elements within each <doc>
        fields = doc.xpath(".//field")
        
        # Iterate through each field and remove it if its text is None or empty
        for field in fields:
            if field.text is None or field.text.strip() == "None":
                field.getparent().remove(field)  # Remove the field if it has a None or empty value

    # Convert the modified XML tree back to a string
    cleaned_xml_data = ET.tostring(root, pretty_print=True, encoding='unicode')
    return cleaned_xml_data


In [113]:
# Usage
filename = '254'
directory = f'/home/sbasir/Thesis/Thesis/EDP/misc/test2'  # Change this to your directory containing RDF/XML files
solr_xml_data, alt_data = parse_rdf_files(directory, filename)

IS NOT TRANSLATED
sv
content is fine
passed checks
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
idk bruh : True
data type is:  <class 'dict'>
{'europeana_id': '/91682/vm_objekt_164771', 'dcterms:modified': '2018-05-07T13:17:28.434Z', 'edm:type': 'IMAGE', 'contentTier': '2', 'metadataTier': '0', 'provided_data': {'edm:dataProvider': {'sv': ['Vänermuseet'], 'en': ['Lake Vänern Museum']}, 'edm:intermediateProvider': {}, 'edm:provider': {'sv': ['K-samsök'], 'en': ['Swedish Open Cultural Heritage']}, 'dc:contributor': None, 'dc:coverage': None, 'dc:creator': {'default': ['Ollson, August [10:01]']}, 'dc:date': {'default': ['1900-1910']}, 'dc:description': {'default': ['Familjen Isaksson utanför hemmet. Fanjunkare Jakob närmast vänster. B

In [114]:
for doc in alt_data:
    cleaned_doc = remove_none(doc)
    print(cleaned_doc)

<add>
  <field name="europeana_id">/91682/vm_objekt_164771</field>
  <field name="timestamp_update">2018-05-07T13:17:28.434Z</field>
  <field name="edm:type">IMAGE</field>
  <field name="content_tier">2</field>
  <field name="metadata_tier">0</field>
  <provided_data>
    <field name="edm:dataProvider">
      <value lang="sv">Vänermuseet</value>
      <value lang="en">Lake Vänern Museum</value>
    </field>
    <field name="edm:provider">
      <value lang="sv">K-samsök</value>
      <value lang="en">Swedish Open Cultural Heritage</value>
    </field>
    <field name="dc:creator">
      <value>Ollson, August [10:01]</value>
    </field>
    <field name="dc:date">
      <value>1900-1910</value>
    </field>
    <field name="dc:description">
      <value>Familjen Isaksson utanför hemmet. Fanjunkare Jakob närmast vänster. Bredvid honom är Åke. Bakom bordet är Stellan och Arvid och längst ut åt vänster är Carolina med med Anders i knät. Gerda sitter framför bordet.</value>
      <value>F</

In [13]:
for doc in solr_xml_data:
    cleaned_doc = remove_none(doc)
    print(cleaned_doc)

<add>
            <doc>
                <field name="europeana_id">/109/https___hispana_mcu_es_lod_oai_bvpb_mcu_es_396852_ent0</field>
                <field name="timestamp_update">2023-01-16T08:38:14.330Z</field>
                <field name="edm_type">TEXT</field>
                <field name="content_tier">4</field>
                <field name="metadata_tier">B</field>
                <provided_data>
                    <field name="data_provider">Virtual Library of Bibliographical Heritage</field>
                    <field name="provider">Hispana</field>
                    <field name="dc_creator">{'es': ['Cistercienses']}</field>
                    <field name="dc_date">{'default': ['1150-1250']}</field>
                    <field name="dc_description">{'es': ['Los dos primeros cuadernos fueron añadidos posteriormente, con materiales de los siglos XIII-XV; hay añadidos de mano gótica en otras partes del volumen', 'Copia digital. Madrid : Ministerio de Cultura. Subdirección Gener

In [48]:
# solr_xml_data = remove_duplicates(solr_xml_data)
output_file = f'/home/sbasir/Thesis/Thesis/EDP/misc/testing/{filename}'
write_data(solr_xml_data, output_file)